## 01 — GeoJSON Structure

In the previous lesson we loaded plain JSON and navigated it as Python dicts and lists.

**GeoJSON** is still just JSON — the same rules apply. The difference is that GeoJSON follows a specific convention about what the keys mean. Once you learn that convention, every GeoJSON file you encounter will look familiar.

The example file for this lesson is `data/meteorites.geojson` — a dataset of meteorite landing sites around the world.

## The GeoJSON Shape

Every valid GeoJSON file is built from three nested layers:

```
FeatureCollection          ← the whole file
└── features: [ ... ]      ← a list of Feature objects
    └── Feature            ← one geographic thing
        ├── type: "Feature"
        ├── geometry       ← WHERE it is
        │   ├── type: "Point"
        │   └── coordinates: [lon, lat]
        └── properties     ← WHAT it is
            ├── name: "Aarhus"
            ├── mass: 720
            └── year: 1951
```

**Critical detail: coordinates are `[longitude, latitude]`, not `[lat, lon]`.**  
This is backwards from how most people say it out loud. It trips up everyone the first time.

## Loading the File

In [1]:
import json
from pathlib import Path

path = Path("data/meteorites.geojson")
data = json.loads(path.read_text())

# inspect the top-level structure
print(type(data))           # <class 'dict'>
print(data.keys())          # dict_keys(['type', 'features'])
print(data["type"])         # FeatureCollection
print(len(data["features"])) # number of meteorite records

<class 'dict'>
dict_keys(['type', 'features'])
FeatureCollection
31963


## Accessing a Single Feature

Unlike `latex_colors.json` where the top level was a list, here `data` is a dict and the list lives at `data["features"]`.

In [ ]:
feature = data["features"][0]

print(feature["type"])                        # Feature
print(feature["properties"]["name"])          # Aarhus
print(feature["properties"]["mass"])          # 720
print(feature["properties"]["year"])          # 1951
print(feature["geometry"]["type"])            # Point
print(feature["geometry"]["coordinates"])     # [10.23333, 56.18333]

## Unpacking Coordinates

`coordinates` is an array: `[longitude, latitude]`.  
Index `0` is longitude (east/west), index `1` is latitude (north/south).

In [ ]:
coords = feature["geometry"]["coordinates"]

lon = coords[0]   # 10.23333
lat = coords[1]   # 56.18333

print(f"lon: {lon},  lat: {lat}")

## Traversal — Loop Over All Features

In [ ]:
for feature in data["features"]:
    props = feature["properties"]
    coords = feature["geometry"]["coordinates"]
    print(f"{props['name']:<30} mass: {props['mass']:>10}  lon: {coords[0]}")

## Traversal — Look Up by Property Value

In [ ]:
match = next(
    (f for f in data["features"] if f["properties"].get("name") == "Acapulco"),
    None
)

if match:
    print(match["properties"])
    print(match["geometry"]["coordinates"])

## Traversal — Collect One Field Across All Features

In [2]:
features = data["features"]

# all names
names = [f["properties"]["name"] for f in features]

# all (lon, lat) pairs
points = [f["geometry"]["coordinates"] for f in features]

# all masses — using .get() with a default since a field might be missing
masses = [f["properties"].get("mass", 0) for f in features]

print(names[:5])
print(points[:5])
print(masses[:5])

['Aarhus', 'Abee', 'Adzhi-Bogdo (stone)', 'Acapulco', 'Achiras']
[[10.23333, 56.18333], [-113.0, 54.21667], [95.16667, 44.83333], [-99.9, 16.88333], [-64.95, -33.16667]]
[720, 107000, 910, 1914, 780]


## GeoJSON vs Plain JSON — Key Differences

| | Plain JSON (`latex_colors.json`) | GeoJSON (`meteorites.geojson`) |
|---|---|---|
| Top-level type | array `[...]` | object `{...}` |
| How to get the list | `colors` directly | `data["features"]` |
| What each item is | an arbitrary object | always a `Feature` |
| Location data | none | `geometry.coordinates` |
| Attribute data | all keys at top level | lives inside `properties` |
| Coordinate order | n/a | `[lon, lat]` — not `[lat, lon]` |

The traversal pattern is the same — the structure just has a predictable shape every time.

## Exercise A

How many meteorites in this dataset have a recorded mass (non-`None`, non-zero)?

Use `.get("mass")` to safely access the field — some features may not have it.

In [8]:
features = data["features"]

# Count how many features have a non-None, non-zero mass value
# Your code here
mass_count = sum(1 for f in features if (f["properties"].get("mass")))
print(mass_count)

31826


## Exercise B

Find all meteorites that fell **after the year 2000**. Print their names and years.

Use `.get("year")` with a default since some entries may be missing that field.

In [16]:
features = data["features"]

# Find all meteorites that fell after year 2000
# Print their names and years
# Your code here
new_met_falls = [f for f in features if f["properties"].get("year") and int(f["properties"].get("year",0)) > 2000]
for f in new_met_falls: print(f["properties"]["name"], f["properties"]["year"])

Alby sur Chéran 2002
Al Zarnkh 2001
Almahata Sitta 2008
Ash Creek 2009
Bassikounou 2006
Battle Mountain 2012
Benguerir 2004
Beni M'hira 2001
Bensour 2002
Berduc 2008
Berthoud 2004
Bhawad 2002
Boumdeid (2003) 2003
Boumdeid (2011) 2011
Bukhara 2001
Bunburra Rockhole 2007
Buzzard Coulee 2008
Cali 2007
Carancas 2007
Chelyabinsk 2013
Chergach 2007
Daule 2008
Devgaon 2001
Dergaon 2001
Didim 2007
Grimsby 2009
Hiroshima 2003
Hoima 2003
Huaxi 2010
Jesenice 2009
Jodiya 2006
Kaprada 2004
Kasauli 2003
Kavarpura 2006
Kemer 2008
Kendrapara 2003
Kilabo 2002
Košice 2010
Lorton 2010
Mahadevpur 2007
Maigatari-Danduma 2004
Maribo 2009
Maromandia 2002
Mifflin 2010
Moss 2006
Neuschwanstein 2002
New Orleans 2003
Orlando 2004
Ouadangou 2003
Oum Dreyga 2003
Park Forest 2003
Pleşcoi 2008
Puerto Lápice 2007
Red Canyon Lake 2007
San Michele 2002
Santa Lucia (2008) 2008
Sołtmany 2011
Sulagiri 2008
Sutter's Mill 2012
Tamdakht 2008
Thika 2011
Thuathe 2002
Tissint 2011
Varre-Sai 2010
Villalbeto de la Peña 2004
Werda

## Exercise C

Find the **3 northernmost** meteorites — those with the highest latitude. Print their names and latitudes, sorted from north to south.

Hint: latitude is `coordinates[1]`.

In [ ]:
features = data["features"]

# Find the 3 northernmost meteorites — highest latitude (index 1 of coordinates)
# Sort and print their names and latitudes
# Your code here
lats = [(f["geometry"]["coordinates"][1], f["properties"]["name"]) for f in features]
lats.sort(reverse=True)
print(lats[:3])

[(81.16667, 'Ryder Gletcher'), (76.53333, 'Thule'), (76.13333, 'Cape York')]




## Check Your Understanding

The meteorite `"Abee"` landed somewhere in Canada. Using the loaded `data` object, write code that prints its **latitude** and the **year** it fell.

```python
# your answer here
match = next(
    (f for f in data["features"] if f["properties"].get("name") == "Abee"),
    None
)
print(match["properties"]["name"])
print(match["geometry"]["coordinates"][1])
```



---

In [ ]:
match = next(
    (f for f in data["features"] if f["properties"].get("name") == "Abee"),
    None
)
print(match["properties"]["name"])
print(match["geometry"]["coordinates"][1])

Abee
54.21667


## Next

In [02 — Feature Collections](./02-Feature_Collections.ipynb), we go beyond reading and start building and modifying GeoJSON collections from scratch.